In [1]:
import source.helpers
import biolib
from pathlib import Path
import pandas as pd

from source import analysis, data, predict, regions
from source import alphamissense as am

C:\Users\domin\anaconda3\envs\GLUT\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


2025-08-30 18:13:34,952 | INFO : Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
2025-08-30 18:13:34,954 | INFO : NumExpr defaulting to 16 threads.


In [2]:
# some libraries that might be neeeded
#pip install pymissense
#pip3 install --upgrade pybiolib 
#pip install gunicorn

In [3]:
proteins = {
'GLUT1':'P11166',
'GLUT2':'P11168',
'GLUT3':'P11169',
#'GLUT4':'P14672',
#'GLUT5':'P22732',
#'GLUT6':'Q9UGQ3',
#'GLUT7':'Q6PXP3',
#'GLUT8':'Q9NY64',
#'GLUT9':'Q9NRM0',
#'GLUT10':'O95528',
#'GLUT11':'Q9BYW1',
#'GLUT12':'Q8TD20',
#'GLUT13':'Q96QE2',
#'GLUT14':'Q8TDB8' 
}

## get data

In [4]:
RES_DIR = Path.cwd() / 'results'
DATA_DIR = Path.cwd() / 'data'

#if results and data direcotries not exists create them
Path.mkdir(RES_DIR, exist_ok=True)
Path.mkdir(DATA_DIR, exist_ok=True)

In [5]:
#import pdb and fasta files
PDB_DIR = DATA_DIR / 'pdb'
FAS_DIR = DATA_DIR / 'fasta'

Path.mkdir(PDB_DIR, exist_ok=True)
Path.mkdir(FAS_DIR, exist_ok=True)
for up_id in proteins.values():
    
    data.get_pdb(up_id, PDB_DIR / f'{up_id}.pdb')
    data.get_fasta(up_id,FAS_DIR / f'{up_id}.fasta')

## get predictions for proteins

In [6]:
alpha_missense_tsv = DATA_DIR / 'AlphaMissense_aa_substitutions.tsv'
data.get_alphamissense(alpha_missense_tsv)

Download complete, unzipping...
AlphaMissense data downloaded


In [7]:
# run pymissense for all proteins and generate csv of aminoacid pathogenicities

PYM_DIR = RES_DIR / 'pymissense'
Path.mkdir(PYM_DIR, exist_ok=True)

for up_id in proteins.values():
    pdb_file = PDB_DIR / f'{up_id}.pdb'
    
    print(f'running PyMissense for {up_id}')
    predict.pymissense(alpha_missense_tsv, pdb_file, up_id, PYM_DIR)

running PyMissense for P11166
AlphaMissense prediction for P11166 already exists
running PyMissense for P11168
AlphaMissense prediction for P11168 already exists
running PyMissense for P11169
AlphaMissense prediction for P11169 already exists


## get protein regions

In [8]:
reg_prefix = {'all':'all',
            'intracellular':'I',
            'extracellular':'O',
            'membrane':'M',
            'binding pocket':'bp',
            'lining residues':'lr',
            'lining residues outside pocket':'nobp_lr'}

### all

In [9]:
REG_DIR = RES_DIR / 'regions'

In [10]:
for up_id in proteins.values():
    regions.all_residues(pdb_file = DATA_DIR / f'pdb/{up_id}.pdb', out_file = REG_DIR/ f'{reg_prefix['all']}_{up_id}.csv')

### intracellular, extracellular and membrane regions

In [11]:
#get annotations of AA positions relative to the membrane using 
deeptmhmm = biolib.load('DTU/DeepTMHMM')
for up_id in proteins.values():
    predict.deepTMHMM(up_id,deeptmhmm)

2025-08-30 18:21:28,544 | INFO : Loaded project DTU/DeepTMHMM:1.0.44
DeepTMHMM prediction for P11166 already exists
DeepTMHMM prediction for P11168 already exists
DeepTMHMM prediction for P11169 already exists


In [12]:
#processing of depptmhmm results
##doplnit prefixy
for up_id in proteins.values():
    regions.membrane_residues(deeptmhmm_file = RES_DIR / f'deeptmhmm/{up_id}.3line', out_dir = REG_DIR, identifier = up_id)

### binding pockets, lining residues and nonpocket binding residues

In [13]:
# The amino acid residues lining the pores of individual proteins were calculated using: https://mole.upol.cz/ -LINING RESIDUES
# The amino acid residues framing the protein binding site were calculated using: https://prankweb.cz/ - BINDING PLACE
# The results were transcribed into an .xlsx document. Saved here /GLUT project documentation/Binding places and lining residues/Excel files/
# After transcription, a third column was created, where only those amino acid residues that were not part of the binding site but framed the protein pore were transcribed. -BINDING PLACE-LINING RESIDUES

In [14]:
#this step is hard to automatize
#uses csv file from PrankWeb
for up_id in proteins.values():
    regions.binding_pockets(prankweb_csv = DATA_DIR / f'prankweb/{up_id}.csv', out_file = REG_DIR/ f'{reg_prefix['binding pocket']}_{up_id}.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\domin\\Desktop\\GLUT\\data\\prankweb\\P11166.csv'

In [ ]:
##unziper na mole online
#for glut in proteins.keys():
#    file_path = DATA_DIR / f'moleonline/{glut}.zip'
#    output = DATA_DIR / f'moleonline/{proteins[glut]}.json'
#    source.helpers.mole_zip2json(file_path, output)

In [ ]:
#lining residues
#from manually obrained JSON files from mole online
for up_id in proteins.values():
    regions.lining_residues(moleonline_json = DATA_DIR / f'moleonline/{up_id}.json', out_file = REG_DIR/ f'{reg_prefix['lining residues']}_{up_id}.csv')

In [ ]:
#nonbinding pocket lining residues
for up_id in proteins.values():
    regions.nonbinding_pocket_lining_residues(bp_residues_csv = REG_DIR / f'{reg_prefix['binding pocket']}_{up_id}.csv', l_residues_csv = REG_DIR / f'{reg_prefix['lining residues']}_{up_id}.csv', out_file = REG_DIR / f'{reg_prefix['lining residues outside pocket']}_{up_id}.csv')

## assign pathogenicites

In [ ]:
AM_DIR = RES_DIR / 'pathogenicities/alphamissense'
PP_DIR = RES_DIR / 'pathogenicities/rhapsody'
SI_DIR = RES_DIR / 'pathogenicities/sift'

In [ ]:
#RHAPSODY
#From dowloanded results, we will continue working just with rhapsody-predictions.txt file for each protein. 
import pandas as pd
import numpy as np
import os

input_folder = DATA_DIR / 'rhapsody'
output_folder = PP_DIR

# Create output folder
os.makedirs(output_folder, exist_ok=True)

# Function for processing a single file
def process_file(filename, output_excel):
    columns = [
        'ID', 'residue_label', 'residue_name', 'AA_alt', 'Training', 'Score', 'Prob', 'Class',
        'pathogenicity', 'PolyPhen_class', 'EVmutation_score', 'EVmutation_class'
    ]

    df = pd.read_csv(filename, sep='\s+', names=columns, comment='#', na_values='nan')

    df = df[['residue_label', 'residue_name', 'pathogenicity']].copy()
    df = df.dropna(subset=['pathogenicity'])

    result = df.groupby(['residue_label', 'residue_name']).agg({'pathogenicity': 'mean'}).reset_index()
    result['pathogenicity'] = result['pathogenicity'].round(3)

    result.to_csv(output_excel, index=False)

# Processing all .txt files
for glut in proteins.keys():
    process_file(input_folder / f'{glut}.txt', output_folder / f'all_{proteins[glut]}.csv')
# This can handle just one protein in a row, need to be repeated 14 times for each protein.

In [ ]:
##SIFT
# After SIFT calculations of protein pathogenicity, we processed results subsequently:
# For every protein we dowloanded matrix directly from the web and saved it under /SIFT_protein name.txt/ which can be found in ../GLUT project documentation/Results from SIFT/
data = []

with open(DATA_DIR / "sift/SIFT_GLUT1.txt", 'r') as f:
    for line in f:
        parts = line.strip().split()

        # Skips headers or rows with non-numeric values
        if not parts or not parts[0][0].isdigit():
            continue

        pos_amk = parts[0]
        try:
            dk = float(parts[1])
            values = list(map(float, parts[2:]))
        except ValueError:
            continue  # skips if any value is not a number

        # Extracting position and AMK
        import re
        match = re.match(r"(\d+)([A-Z])", pos_amk)
        if match:
            pos, amk = match.groups()

            # (sum(values) - 1) / 19
            if len(values) == 20:  # sanity check
                ave = (sum(values) - 1) / 19
            else:
                ave = None  # unexpected format

            data.append([pos, amk, dk, ave])

# Creating a DataFrame and saving it
import pandas as pd
df = pd.DataFrame(data, columns=['pos', 'AMK', 'dk', 'ave'])
df.to_csv(SI_DIR / 'glut1.txt', sep='\t', index=False)

# This part is written just for a single protein, so you have to reapeat it 14 times. 
# Thorugh this part we counted averages for each position of aminoacid and saved it under the protein name. Can be found in ../GLUT project documentation/Results from SIFT/Averages/

In [ ]:
for glut in proteins.keys():
    input_file = DATA_DIR / f"sift/SIFT_{glut}.txt"
    output_file = SI_DIR / f"all_{proteins[glut]}.csv"
    names = 'ACDEFGHIKLMNPQRSTVWY'
    active = False
    residue_labels = []
    residue_names = []
    pathogenicities = []
    
    with open(input_file,'r') as f:
        for line in f:
            if active and line.startswith('pos'):
                continue
            elif active:
                l = line.split()
                position = l[0][:-1]
                name = l[0][-1]
                pathos = l[1:]
                c = 0
                for i in range(20):
                    if names[i] != name:
                        c += float(pathos[i])
                avg_pat = c/19
                residue_labels.append(position)
                residue_names.append(name)
                pathogenicities.append(avg_pat)
            elif line.startswith('pos'):
                active = True
    pd.DataFrame(data={'residue_label':residue_labels, 'residue_name':residue_names, 'pathogenicity':pathogenicities}).to_csv(output_file)
                
            

In [ ]:
for up_id in proteins.values():
    am_patho_csv = AM_DIR / f'{reg_prefix['all']}_{up_id}.csv'
    pp_patho_csv = PP_DIR / f'{reg_prefix['all']}_{up_id}.csv'
    si_patho_csv = SI_DIR / f'{reg_prefix['all']}_{up_id}.csv'
    #create file for all
    am.residues(RES_DIR / f'pymissense/{up_id}-edit.pdb', am_patho_csv)
    #create file for rest
    for reg in list(reg_prefix.keys())[1:]:
        analysis.assign_pathogenicity(am_patho_csv, region_csv = REG_DIR / f'{reg_prefix[reg]}_{up_id}.csv',out_file = AM_DIR / f'{reg_prefix[reg]}_{up_id}.csv')
        analysis.assign_pathogenicity(pp_patho_csv, region_csv = REG_DIR / f'{reg_prefix[reg]}_{up_id}.csv',out_file = PP_DIR / f'{reg_prefix[reg]}_{up_id}.csv')
        analysis.assign_pathogenicity(si_patho_csv, region_csv = REG_DIR / f'{reg_prefix[reg]}_{up_id}.csv',out_file = SI_DIR / f'{reg_prefix[reg]}_{up_id}.csv')

## average pathogenicities


In [ ]:
for prefix in reg_prefix.values():
    analysis.average_pathogenicity(patho_dir = AM_DIR, region_prefix = prefix, up_id_list = proteins.values(), out_file = AM_DIR / f'{prefix}_average_patho.csv')
    analysis.average_pathogenicity(patho_dir = PP_DIR, region_prefix = prefix, up_id_list = proteins.values(), out_file = PP_DIR / f'{prefix}_average_patho.csv')
    analysis.average_pathogenicity(patho_dir = SI_DIR, region_prefix = prefix, up_id_list = proteins.values(), out_file = SI_DIR / f'{prefix}_average_patho.csv')


## united pathogenicites

In [ ]:
united_df = pd.read_csv(AM_DIR / f'{reg_prefix['all']}_average_patho.csv')
united_df.columns = ['identifier', 'all']
for reg in list(reg_prefix.keys())[1:]:
    pathogenicities = []
    df = pd.read_csv(AM_DIR / f'{reg_prefix[reg]}_average_patho.csv')
    for index, row in united_df.iterrows():
        identifier = row['identifier']
        pat = df[df['uniprot_id']==identifier]['average_pathogenicity']
        pathogenicities.append(pat.iloc[0])
    united_df.loc[:, reg] = pathogenicities

In [ ]:
#rename to GLUT names
for glut_id in proteins.keys():
    i = united_df[united_df['identifier']==proteins[glut_id]].index
    united_df.at[i.values[0], 'identifier'] = glut_id
united_df
united_df.to_csv(AM_DIR / 'united_averages.csv', index=False)

In [ ]:
#Creating the final heatmap
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Loading file
df = pd.read_csv(AM_DIR / 'united_averages.csv')

# Setting 'filename' as index
df.set_index("identifier", inplace=True)

# Creating a heat map with values
plt.figure(figsize=(10, len(df) * 0.4))
sns.heatmap(
    df,
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor='gray',
    annot=True,
    fmt=".3f"  
)


plt.title("GLUTs pathogenicity profile Alpha Missense", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()


plt.savefig("heatmap_patogenicityPyMissense.png", dpi=300, bbox_inches='tight')

plt.show()